In [ ]:
import pylupnt

print("pylupnt version:", pylupnt.__version__)

In [ ]:
from pylupnt import plasma as pecsimpy
from datetime import datetime, timezone

print(pecsimpy.__file__)

In [ ]:
import inspect

print("Module path:", pecsimpy.__file__)
print("Functions in module:")
for name, obj in inspect.getmembers(pecsimpy, callable):
    if not name.startswith("__"):
        print(" ", name)

In [ ]:
from pylupnt.plasma.kp_loader import update_kp

base_path = pecsimpy.get_plasma_base_path()
print("Base path:", base_path)

# Load latest kp index
update_kp(base_path)

In [ ]:
import numpy as np

year = 2002
doy = 185
hour = 12
minute = 0
sec = 0
dt_cpp = pecsimpy.DateTime(year, doy, hour, minute, sec)

r = 1.014
akp = 0.7
al = 1.107226364
alatr = np.arccos(np.sqrt(r / al))  # [0, pi]
amlt = 23.74

pecsimpy.set_iri_model("IRI2007")

mjd = pecsimpy.datetime_to_mjd(dt_cpp)
tj2000 = pecsimpy.mjd_to_tj2000(mjd)

# IRI options
iri2007option = pecsimpy.IRI2007Option()
iri2007option.R12 = -1.0  # No R12 correction
pecsimpy.set_iri2007_option(iri2007option)

ionoparams = pecsimpy.get_iono_params(tj2000, akp)

# Case 1:
out_cpp = pecsimpy.gcpm_v24(dt_cpp, r, amlt, alatr, akp)
out_fortran = pecsimpy.gcpm_v24_fortran(dt_cpp, r, amlt, alatr, akp)
print("[Case 1] Default case")
print(
    "Year: {} , DOY: {}, Hour: {}, Minute: {}, Sec: {}".format(
        year, doy, hour, minute, sec
    )
)
print("Magnetic latitude (alatr):", alatr)
print("Magnetic longitude (amlt):", amlt)
print("Geocentric Radius (r):", r)
print("F10.7 index:", out_cpp[4], " (from IRI2007Option)", ionoparams[0])
print("Rz index (R12):", out_cpp[5], " (from IRI2007Option)", ionoparams[1])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (C++):", out_cpp[:4])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (Fortran):", out_fortran)

# Case 2:
r += 0.5
out_cpp = pecsimpy.gcpm_v24(dt_cpp, r, amlt, alatr, akp)
out_fortran = pecsimpy.gcpm_v24_fortran(dt_cpp, r, amlt, alatr, akp)
print("\n[Case 2] Change radius")
print(
    "Year: {} , DOY: {}, Hour: {}, Minute: {}, Sec: {}".format(
        year, doy, hour, minute, sec
    )
)
print("Magnetic latitude (alatr):", alatr)
print("Magnetic longitude (amlt):", amlt)
print("Geocentric Radius (r):", r)
print("F10.7 index:", out_cpp[4])
print("Rz index (R12):", out_cpp[5])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (C++):", out_cpp[:4])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (Fortran):", out_fortran)

# Case 3:
dt_cpp.year = 2025
dt_cpp.doy = 1
dt_cpp.hour = 12
dt_cpp.min = 0
dt_cpp.sec = 0
r = 2.2
akp = 6.5
out_cpp = pecsimpy.gcpm_v24(dt_cpp, r, amlt, alatr, akp)
out_fortran = pecsimpy.gcpm_v24_fortran(dt_cpp, r, amlt, alatr, akp)
print("\n[Case 3] Change date and time, radius, and kp index")
print(
    "Year: {} , DOY: {}, Hour: {}, Minute: {}, Sec: {}".format(
        dt_cpp.year, dt_cpp.doy, dt_cpp.hour, dt_cpp.min, dt_cpp.sec
    )
)
print("Magnetic latitude (alatr):", alatr)
print("Magnetic longitude (amlt):", amlt)
print("Geocentric Radius (r):", r)
print("F10.7 index:", out_cpp[4])
print("Rz index (R12):", out_cpp[5])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (C++):", out_cpp[:4])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (Fortran):", out_fortran)

# Case4: Free kp index
akp = pecsimpy.get_kp_index(dt_cpp)
out_cpp = pecsimpy.gcpm_v24(dt_cpp, r, amlt, alatr, akp)
out_fortran = pecsimpy.gcpm_v24_fortran(dt_cpp, r, amlt, alatr, akp)
print("\n[Case 4] Free kp index")
print(
    "Year: {} , DOY: {}, Hour: {}, Minute: {}, Sec: {}".format(
        dt_cpp.year, dt_cpp.doy, dt_cpp.hour, dt_cpp.min, dt_cpp.sec
    )
)
print("Magnetic latitude (alatr):", alatr)
print("Magnetic longitude (amlt):", amlt)
print("Geocentric Radius (r):", r)
print("Free kp index (akp):", akp)
print("F10.7 index:", out_cpp[4])
print("Rz index (R12):", out_cpp[5])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (C++):", out_cpp[:4])
print("Total Ne,  H+,  He+, O+  [cm^-3] (Fortran):", out_fortran)

# Case 5: Test with
iri2007option.R12 = 1.0  # No R12 correction
pecsimpy.set_iri2007_option(iri2007option)
out_cpp = pecsimpy.gcpm_v24(dt_cpp, r, amlt, alatr, akp)
out_fortran = pecsimpy.gcpm_v24_fortran(dt_cpp, r, amlt, alatr, akp)
print("\n[Case 5] Change R12 correction to 1.0")
print(
    "Year: {} , DOY: {}, Hour: {}, Minute: {}, Sec: {}".format(
        dt_cpp.year, dt_cpp.doy, dt_cpp.hour, dt_cpp.min, dt_cpp.sec
    )
)
print("Magnetic latitude (alatr):", alatr)
print("Magnetic longitude (amlt):", amlt)
print("Geocentric Radius (r):", r)
print("Free kp index (akp):", akp)
print("F10.7 index:", out_cpp[4])
print("Rz index (R12):", out_cpp[5])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (C++):", out_cpp[:4])
print("Total Ne,  H+,  He+, O+  [cm^-3] (Fortran):", out_fortran)

# case 6: use IRI 2020 model
pecsimpy.set_iri_model("IRI2020")
out_cpp = pecsimpy.gcpm_v24(dt_cpp, r, amlt, alatr, akp)
out_fortran = pecsimpy.gcpm_v24_fortran(dt_cpp, r, amlt, alatr, akp)
print("\n[Case 6] Use IRI 2020 model")
print(
    "Year: {} , DOY: {}, Hour: {}, Minute: {}, Sec: {}".format(
        dt_cpp.year, dt_cpp.doy, dt_cpp.hour, dt_cpp.min, dt_cpp.sec
    )
)
print("Magnetic latitude (alatr):", alatr)
print("Magnetic longitude (amlt):", amlt)
print("Geocentric Radius (r):", r)
print("Free kp index (akp):", akp)
print("F10.7 index:", out_cpp[4])
print("Rz index (R12):", out_cpp[5])
print("Total Ne,  H+,  He+,  O+  [cm^-3] (C++ + IRI2020):", out_cpp[:4])
print("Total Ne,  H+,  He+, O+  [cm^-3] (Fortran + IRI2007):", out_fortran)